# 02: Cyclical Feature Engineering, Sine/Cosine Transforms & Optuna Tuning

**Track 04: Regression & Continuous Prediction** | *Tensorbox AI/ML Production Curriculum*

---
### Overview & Objectives
Model periodicity in hourly/seasonal demand (Bike Sharing dataset) using Trigonometric transformations, Fourier terms, and Optuna automated Bayesian hyperparameter search.


## 1. Cyclical Encoding: Continuous Periodicity
When modeling hours (0-23) or months (1-12), distance between 23:00 and 00:00 is 1 hour, not 23.

Transforming raw cyclic index $t$ with period $T$:
$$x_{\sin} = \sin\left(\frac{2\pi t}{T}\right), \quad x_{\cos} = \cos\left(\frac{2\pi t}{T}\right)$$

In [ ]:
import os
import sys
from pathlib import Path

for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (p / "utils").exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        break

from utils.data_loader import load_dataset

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

df = load_dataset("bike_sharing")
print(f"Bike Sharing Dataset: {df.shape}")

# Encode hour periodicity
if "hour" in df.columns:
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24.0)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24.0)

# Encode month periodicity
if "month" in df.columns:
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12.0)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12.0)

print(df[["hour", "hour_sin", "hour_cos"]].head())

## 2. Gradient Boosting Demand Regression
Train a Gradient Boosting model to forecast hourly bicycle rentals.

In [ ]:
target = "count" if "count" in df.columns else df.columns[-1]
feat_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != target]

X = df[feat_cols]
y = df[target]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

gbr = GradientBoostingRegressor(n_estimators=100, learning_rate=0.08, max_depth=5, random_state=42)
gbr.fit(X_tr, y_tr)
preds = gbr.predict(X_te)

rmse = np.sqrt(mean_squared_error(y_te, preds))
r2 = r2_score(y_te, preds)

print("=== Demand Regression Model Results ===")
print(f"Test RMSE    : {rmse:.2f}")
print(f"Test R² Score: {r2:.4f}")